In [1]:
!pip uninstall -y kernels transformers
!pip install transformers[torch] accelerate huggingface_hub av decord

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 51.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 85.0 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you 

In [2]:
import os
import shutil
import re
import torch
from huggingface_hub import HfApi, hf_hub_download, login
from kaggle_secrets import UserSecretsClient
from transformers import Sam3VideoModel, Sam3VideoProcessor
from transformers.video_utils import load_video
from accelerate import Accelerator
import gc

# ================= CONFIGURATION =================
REPO_ID = "hannusia123123/drone_dataset"
BATCH_SIZE = 50
PROMPT_TEXT = "military drone, UAV, delta-wing flying object, cross-wing drone"
VIDEO_EXT = ('.mp4', '.mov', '.m4v')
IGNORE_LIST = ["wing_rgb_sky_428.mp4", "wing_rgb_sky_376.mp4", "wing_rgb_sky_372.mp4"]
# =================================================

def setup_huggingface():
    user_secrets = UserSecretsClient()
    login(token=user_secrets.get_secret("HF_TOKEN"))
    return HfApi()

def get_processing_state(api):
    files_in_repo = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
    
    log_file_repo_path = "cvat_batches/processed_log.txt"
    # Визначаємо абсолютний шлях до локального логу
    log_file_local = os.path.join("/kaggle/working/", "processed_log.txt")
    processed_videos = set()
    
    if log_file_repo_path in files_in_repo:
        # Завантажуємо файл
        path_to_downloaded = hf_hub_download(repo_id=REPO_ID, filename=log_file_repo_path, repo_type="dataset")
        # Копіюємо його в зручне для нас місце, щоб точно знати шлях
        shutil.copy(path_to_downloaded, log_file_local)
        
        with open(log_file_local, "r") as f:
            processed_videos = set(f.read().splitlines())
    else:
        # Створюємо файл, якщо його немає
        with open(log_file_local, 'w') as f:
            pass
        
    # Послідовно перевіряємо спліти: як тільки знаходимо необроблені файли — зупиняємось на ньому
    for current_split in ["train", "val", "test"]:
        all_split_videos = [f for f in files_in_repo if f.startswith(f"{current_split}/") and f.endswith(VIDEO_EXT)]
        queue = [v for v in all_split_videos if v not in processed_videos]
        
        if queue:
            # Шукаємо останній номер батчу САМЕ ДЛЯ ЦЬОГО СПЛІТУ
            batch_folders = [f for f in files_in_repo if f.startswith(f"cvat_batches/{current_split}/batch_")]
            batch_nums = []
            for f in batch_folders:
                match = re.search(r'batch_(\d+)', f)
                if match: batch_nums.append(int(match.group(1)))
                    
            next_batch_num = max(batch_nums) + 1 if batch_nums else 1
            return current_split, queue, next_batch_num, log_file_local
            
    # Якщо дійшли сюди, значить всі три спліти порожні (все розмічено)
    return None, [], 0, log_file_local

def create_cvat_zip(outputs_per_frame, video_frames, video_filename, output_dir):
    """Creates an archive in MOT format for tracking in CVAT"""
    base_dir = os.path.join(output_dir, "cvat_mot_tracking")
    gt_dir = os.path.join(base_dir, "gt")
    os.makedirs(gt_dir, exist_ok=True)
    
    # Файл з назвами класів
    with open(os.path.join(base_dir, "labels.txt"), "w") as f:
        f.write("drone\n")
        
    # Файл з трекінгом
    with open(os.path.join(gt_dir, "gt.txt"), "w") as f:
        for frame_idx, data in outputs_per_frame.items():
            mot_frame_idx = frame_idx + 1 # MOT зазвичай починає з 1
            if "boxes" in data and len(data["boxes"]) > 0:
                for i, box in enumerate(data["boxes"]):
                    track_id = i + 1
                    x1, y1, x2, y2 = box.tolist()
                    width = x2 - x1
                    height = y2 - y1
                    # Формат: frame, id, x, y, w, h, active, class, visibility
                    f.write(f"{mot_frame_idx},{track_id},{x1:.2f},{y1:.2f},{width:.2f},{height:.2f},1,1,1\n")
                    
    zip_path = os.path.join(output_dir, os.path.splitext(os.path.basename(video_filename))[0] + "_mot")
    return shutil.make_archive(zip_path, 'zip', base_dir)

def main():
    api = setup_huggingface()
    current_split, video_queue, batch_num, log_file = get_processing_state(api)
    
    if not current_split:
        print("🎉 All videos (Train, Val, Test) have been fully annotated! The task is complete.")
        return
        
    current_batch = video_queue[:BATCH_SIZE]
    batch_name = f"batch_{batch_num:03d}"
    print(f"📁 Current split: {current_split.upper()}")
    print(f"📦 Creating {batch_name}. Videos in the queue for this split: {len(current_batch)}")
    
    device = Accelerator().device
    print("⏳ Loading SAM 3...")
    model = Sam3VideoModel.from_pretrained("facebook/sam3", torch_dtype=torch.float16).to(device)
    processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")
    
    local_batch_dir = f"/kaggle/working/{batch_name}"
    os.makedirs(local_batch_dir, exist_ok=True)
    processed_in_this_run = []

    for remote_video_path in current_batch:
        video_filename = os.path.basename(remote_video_path)

        if video_filename in IGNORE_LIST:
            print(f"⚠️ Skipping {video_filename} (in IGNORE_LIST due to OOM)")
            # Записуємо його в лог, ніби обробили, щоб скрипт більше не намагався його взяти
            processed_in_this_run.append(remote_video_path)
            continue
        
        print(f"\n🎥 Processing: {video_filename}")
        
        local_vid_path = hf_hub_download(repo_id=REPO_ID, filename=remote_video_path, repo_type="dataset", local_dir="/kaggle/working/")
        
        video_frames = load_video(local_vid_path)[0]
        inference_session = processor.init_video_session(
            video=video_frames, inference_device=device, processing_device="cpu", video_storage_device="cpu", dtype=torch.float16
        )
        processor.add_text_prompt(inference_session=inference_session, text=PROMPT_TEXT)
        
        outputs_per_frame = {}
        for model_outputs in model.propagate_in_video_iterator(inference_session=inference_session):
            processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
            outputs_per_frame[model_outputs.frame_idx] = processed_outputs
            
        create_cvat_zip(outputs_per_frame, video_frames, video_filename, local_batch_dir)
        processed_in_this_run.append(remote_video_path)
        os.remove(local_vid_path)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    print(f"\n🚀 Uploading {batch_name} to Hugging Face...")
    with open(log_file, "a") as f:
        for vid in processed_in_this_run:
            f.write(f"{vid}\n")
            
    # ЗВЕРНИ УВАГУ: Тепер ми пушимо у папку конкретного спліту!
    api.upload_folder(
        folder_path=local_batch_dir,
        path_in_repo=f"cvat_batches/{current_split}/{batch_name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Add SAM3 annotations for {current_split} / {batch_name}"
    )
    
    api.upload_file(
        path_or_fileobj=log_file,
        path_in_repo="cvat_batches/processed_log.txt",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message=f"Update processed log after {current_split}/{batch_name}"
    )
    
    shutil.rmtree(local_batch_dir)
    print(f"✅ Batch {batch_name} has been successfully saved to split {current_split.upper()}.")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if __name__ == "__main__":
    main()

processed_log.txt: 0.00B [00:00, ?B/s]

📁 Current split: TEST
📦 Creating batch_004. Videos in the queue for this split: 9
⏳ Loading SAM 3...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

...

test/wing_thermal_sky_591.mp4:   0%|          | 0.00/1.16M [00:00<?, ?B/s]


🚀 Uploading batch_004 to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Batch batch_004 has been successfully saved to split TEST.
